# BoltzGen Nanobody Scoring
**Target:** P05231 (IL-6)  
**Source:** [github.com/schneider-weave/boltz_scoring](https://github.com/schneider-weave/boltz_scoring)

Pipeline: **design/structure generation -> folding -> analysis -> filtering**. `--skip_inverse_folding` skips only inverse folding; it does not make folding read YAML files directly.

> **Kaggle settings required:** Accelerator = GPU, Internet = ON for default checkpoint downloads. Run the one-input smoke test first, then set `RUN_ALL = True`.

## 1. Clone repo & install

In [ ]:
import os

REPO_URL = "https://github.com/schneider-weave/boltz_scoring"
REPO_DIR = "/kaggle/working/boltz_scoring"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready:", REPO_DIR)

In [ ]:
%%bash
# CUDA-compatible torch for Kaggle (CUDA 12.x)
pip install -q torch --index-url https://download.pytorch.org/whl/cu121

# cuequivariance wheels required by boltzgen
pip install -q \
    cuequivariance-ops-torch-cu12 \
    cuequivariance-torch \
    cuequivariance-ops-cu12

# Install boltzgen from local source (bittensor-free)
pip install -q -e /kaggle/working/boltz_scoring/boltzgen

echo "Done"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!boltzgen -v

## 2. Paths & configuration

In [ ]:
from pathlib import Path

REPO_DIR       = Path("/kaggle/working/boltz_scoring")
SCORING_INPUTS = REPO_DIR / "scoring_inputs"
OUTPUT_DIR     = Path("/kaggle/working/scoring_results")
CACHE_DIR      = Path("/kaggle/working/cache")
MSA_PATH       = REPO_DIR / "msa_files" / "P05231.a3m"

# Safety switch: keep False until one YAML completes successfully.
RUN_ALL = False

# Optional custom checkpoints mounted from Kaggle Input.
# Examples:
# CUSTOM_DESIGN_CHECKPOINTS = [Path("/kaggle/input/my-boltzgen-model/design.ckpt")]
# CUSTOM_FOLDING_CHECKPOINT = Path("/kaggle/input/my-boltzgen-model/folding.ckpt")
CUSTOM_DESIGN_CHECKPOINTS = []
CUSTOM_FOLDING_CHECKPOINT = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

assert SCORING_INPUTS.exists(), f"Missing scoring input directory: {SCORING_INPUTS}"
assert MSA_PATH.exists(), f"Missing target MSA: {MSA_PATH}"

yaml_files = sorted(SCORING_INPUTS.glob("*.yaml"))
assert yaml_files, f"No YAML files found in {SCORING_INPUTS}"

# Make the MSA path absolute inside Kaggle so path resolution cannot depend on cwd.
for yf in yaml_files:
    txt = yf.read_text()
    fixed = txt.replace("../msa_files/P05231.a3m", str(MSA_PATH))
    if fixed != txt:
        yf.write_text(fixed)

custom_design = [Path(p) for p in CUSTOM_DESIGN_CHECKPOINTS]
custom_folding = Path(CUSTOM_FOLDING_CHECKPOINT) if CUSTOM_FOLDING_CHECKPOINT else None
for ckpt in custom_design + ([custom_folding] if custom_folding else []):
    assert ckpt.exists(), f"Custom checkpoint does not exist: {ckpt}"

INPUT_SPEC = SCORING_INPUTS if RUN_ALL else yaml_files[0]
print(f"Input YAMLs      : {len(yaml_files)}")
print(f"Selected input   : {INPUT_SPEC}")
print(f"MSA exists       : {MSA_PATH.exists()} ({MSA_PATH})")
print(f"Output dir       : {OUTPUT_DIR}")
print(f"Cache dir        : {CACHE_DIR}")
print(f"Custom design ckpt(s): {[str(p) for p in custom_design] or 'default Hugging Face'}")
print(f"Custom folding ckpt : {str(custom_folding) if custom_folding else 'default Hugging Face'}")

print(f"\nSample input ({yaml_files[0].name}):")
print(yaml_files[0].read_text())

## 3. Download model weights (~6 GB, cached after first run)

In [ ]:
import subprocess

artifacts = ["moldir"]
if not custom_design:
    artifacts.append("design-diverse")
if custom_folding is None:
    artifacts.append("folding")

cmd = ["boltzgen", "download", *artifacts, "--cache", str(CACHE_DIR)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Downloaded/verified artifacts:", artifacts)

## 4. Run BoltzGen Pipeline

This runs the correct dependency order for this repo: YAML input -> `intermediate_designs/` -> refolding -> analysis -> filtering.

`--skip_inverse_folding` skips inverse folding only. The first design/structure-generation step is still required because folding reads structures from `intermediate_designs/`, not directly from YAML.

In [ ]:
import subprocess, sys

run_cmd = [
    "boltzgen", "run", str(INPUT_SPEC),
    "--output", str(OUTPUT_DIR),
    "--protocol", "nanobody-anything",
    "--cache", str(CACHE_DIR),
    "--skip_inverse_folding",
    "--num_designs", "1",
    "--budget", "20",
    "--reuse",
    "--devices", "1",
    "--num_workers", "0",
    "--use_kernels", "false",
    # Kaggle is memory constrained; keep CPU analysis modest.
    "--config", "analysis", "num_processes=2", "liability_modality=antibody",
    # Real nanobodies often contain framework cysteines, so do not hard-filter Cys.
    "--config", "filtering", "filter_cysteine=false", "modality=antibody",
]

if custom_design:
    run_cmd += ["--design_checkpoints", *[str(p) for p in custom_design]]
if custom_folding is not None:
    run_cmd += ["--folding_checkpoint", str(custom_folding)]

print("Running:", " ".join(run_cmd))
print("-" * 80)
subprocess.run(run_cmd, check=True)
print("\n[OK] BoltzGen pipeline completed")

In [ ]:
# Verify the actual output layout used by this BoltzGen fork.
from boltzgen.data import const

DESIGN_DIR = OUTPUT_DIR / "intermediate_designs"
REFOLD_CIF_DIR = DESIGN_DIR / const.refold_cif_dirname
FOLD_NPZ_DIR = DESIGN_DIR / const.folding_dirname
METRICS_CSV = DESIGN_DIR / "aggregate_metrics_analyze.csv"
FINAL_DIR = OUTPUT_DIR / "final_ranked_designs"

original_cifs = sorted(DESIGN_DIR.glob("*.cif")) if DESIGN_DIR.exists() else []
original_npzs = sorted(DESIGN_DIR.glob("*.npz")) if DESIGN_DIR.exists() else []
refold_cifs = sorted(REFOLD_CIF_DIR.glob("*.cif")) if REFOLD_CIF_DIR.exists() else []
fold_npzs = sorted(FOLD_NPZ_DIR.glob("*.npz")) if FOLD_NPZ_DIR.exists() else []

print(f"Design dir         : {DESIGN_DIR} (exists={DESIGN_DIR.exists()})")
print(f"Original CIF files : {len(original_cifs)}")
print(f"Original NPZ files : {len(original_npzs)}")
print(f"Refold CIF files   : {len(refold_cifs)}")
print(f"Fold NPZ files     : {len(fold_npzs)}")
print(f"Metrics CSV        : {METRICS_CSV} (exists={METRICS_CSV.exists()})")
print(f"Final ranked dir   : {FINAL_DIR} (exists={FINAL_DIR.exists()})")

if not METRICS_CSV.exists():
    print("\n[ERROR] Metrics CSV was not produced. First output files:")
    for p in sorted(OUTPUT_DIR.rglob("*"))[:60]:
        print(" ", p)
    raise FileNotFoundError(METRICS_CSV)

print("\n[OK] Pipeline output layout looks correct")

## 5. Analysis Output

Analysis is already included in the full pipeline above. This section only locates and validates the resulting metrics CSV.

In [ ]:
# No separate analysis run is needed. Re-running only analysis before valid folding
# outputs exist is one of the causes of the original Kaggle errors.
print("Analysis was run as part of the full pipeline above.")

In [ ]:
metrics_csv = METRICS_CSV
if not metrics_csv.exists():
    found = sorted(OUTPUT_DIR.rglob("aggregate_metrics_analyze.csv"))
    metrics_csv = found[0] if found else None

if metrics_csv is None or not metrics_csv.exists():
    print("[ERROR] aggregate_metrics_analyze.csv not found. Output dir contents:")
    for p in sorted(OUTPUT_DIR.rglob("*"))[:60]:
        print(" ", p)
    raise FileNotFoundError("aggregate_metrics_analyze.csv")

print(f"[OK] Metrics CSV found: {metrics_csv}")

## 6. Filtering Output

Filtering is already included in the full pipeline above. This section validates the ranked output directory.

In [ ]:
if FINAL_DIR.exists():
    print(f"[OK] Final ranked directory: {FINAL_DIR}")
    for p in sorted(FINAL_DIR.rglob("*"))[:30]:
        print(" ", p)
else:
    print("[WARNING] final_ranked_designs was not created. Metrics are still available for manual scoring.")

## 7. Load & display results

In [ ]:
import pandas as pd

df = pd.read_csv(metrics_csv)
print(f"Loaded {len(df)} results — columns: {list(df.columns)}")
df.head(3)

In [ ]:
DESIRED_METRICS = [
    "id",
    "delta_sasa_refolded",          # buried surface area, higher is better
    "delta_sasa_original",
    "plip_hbonds_refolded",         # H-bonds at interface, higher is better
    "plip_hbonds",
    "plip_saltbridge_refolded",
    "plip_saltbridge",
    "noncovalents_refolded",
    "noncovalents",
    "design_largest_hydrophobic_patch_refolded",  # lower is better
    "largest_hydrophobic_refolded",
    "plddt",
    "ptm",
    "iptm",
    "design_to_target_iptm",
    "design_ptm",
    "min_design_to_target_pae",     # lower is better
]

available = [c for c in DESIRED_METRICS if c in df.columns]
missing = [c for c in DESIRED_METRICS if c not in df.columns]
if missing:
    print(f"Not available: {missing}")

assert "id" in df.columns, "Metrics CSV must contain an 'id' column"
df_scores = df[available].copy()
df_scores["nanobody_id"] = df_scores["id"].str.extract(r"(design_spec_\d+)", expand=False)
df_scores["original_rank"] = df_scores["id"].str.extract(r"rank=(\d+)", expand=False).astype(float)

# Prefer interface quality metrics. Fall back cleanly depending on what this run produced.
priority = [
    ("delta_sasa_refolded", False),
    ("delta_sasa_original", False),
    ("plip_hbonds_refolded", False),
    ("plip_hbonds", False),
    ("design_to_target_iptm", False),
    ("iptm", False),
    ("min_design_to_target_pae", True),
]
sort_cols = [c for c, _ in priority if c in df_scores.columns]
ascending = [asc for c, asc in priority if c in df_scores.columns]

if not sort_cols:
    raise ValueError(f"No ranking metrics found. Available columns: {list(df.columns)}")

df_ranked = df_scores.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)
df_ranked.index += 1
df_ranked.index.name = "boltzgen_rank"

print(f"\nTop 20 nanobodies (ranked by {sort_cols}):")
df_ranked.head(20)

## 8. Visualize

In [ ]:
import matplotlib.pyplot as plt

plot_metrics = [c for c in [
    "delta_sasa_refolded", "delta_sasa_original",
    "plip_hbonds_refolded", "plip_hbonds",
    "design_largest_hydrophobic_patch_refolded", "plddt", "iptm", "design_to_target_iptm",
] if c in df_ranked.columns]

if not plot_metrics:
    print("No plottable metrics found.")
else:
    fig, axes = plt.subplots(1, len(plot_metrics), figsize=(4 * len(plot_metrics), 4))
    if len(plot_metrics) == 1:
        axes = [axes]

    for ax, metric in zip(axes, plot_metrics):
        ax.hist(df_ranked[metric].dropna(), bins=25, edgecolor="white", color="steelblue")
        ax.set_title(metric.replace("_", "\n"), fontsize=8)
        ax.set_xlabel("score")

    plt.suptitle("BoltzGen Score Distributions - P05231 Nanobodies", fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig("/kaggle/working/score_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
x_col = next((c for c in ["delta_sasa_refolded", "delta_sasa_original"] if c in df_ranked.columns), None)
y_col = next((c for c in ["plip_hbonds_refolded", "plip_hbonds", "design_to_target_iptm", "iptm"] if c in df_ranked.columns), None)

if x_col and y_col:
    fig, ax = plt.subplots(figsize=(7, 5))
    sc = ax.scatter(df_ranked[x_col], df_ranked[y_col],
                    c=df_ranked.index, cmap="viridis_r", alpha=0.8, s=40)
    plt.colorbar(sc, ax=ax, label="boltzgen_rank")
    ax.set_xlabel(f"{x_col}  (higher is better)")
    ax.set_ylabel(f"{y_col}  (higher is better)")
    ax.set_title("Binding Quality - P05231 Nanobodies")
    for i, row in df_ranked.head(5).iterrows():
        ax.annotate(str(row.get("nanobody_id", i)), (row[x_col], row[y_col]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")
    plt.tight_layout()
    plt.savefig("/kaggle/working/sasa_vs_hbonds.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping scatter plot; suitable x/y metrics were not found.")

## 9. Save results

In [ ]:
out_csv = "/kaggle/working/nanobody_boltzgen_scores.csv"
df_ranked.to_csv(out_csv)
print(f"Saved: {out_csv}")

if plot_metrics:
    print("\nSummary:")
    display(df_ranked[plot_metrics].describe().round(3))
else:
    print("No summary metrics available to describe.")

In [ ]:
# Attach original sequences from input YAMLs.
import yaml as pyyaml

seq_map = {}
for yf in sorted(SCORING_INPUTS.glob("*.yaml")):
    data = pyyaml.safe_load(yf.read_text())
    for entity in data.get("entities", []):
        if entity.get("protein", {}).get("id") == "B":
            seq_map[yf.stem] = entity["protein"]["sequence"]

def lookup_sequence(row):
    row_id = str(row.get("id", ""))
    nb_id = str(row.get("nanobody_id", ""))
    return next((v for k, v in seq_map.items() if k in row_id or (nb_id and nb_id in k)), None)

df_ranked["sequence"] = df_ranked.apply(lookup_sequence, axis=1)

sasa_col = next((c for c in ["delta_sasa_refolded", "delta_sasa_original"] if c in df_ranked.columns), None)
hb_col = next((c for c in ["plip_hbonds_refolded", "plip_hbonds"] if c in df_ranked.columns), None)

# Save top 10 as FASTA.
fasta_out = "/kaggle/working/top10_boltzgen_scored.fasta"
with open(fasta_out, "w") as f:
    for rank, row in df_ranked.head(10).iterrows():
        nb_id = row.get("nanobody_id") or row.get("id") or f"nb_{rank}"
        sasa = f"{row[sasa_col]:.2f}" if sasa_col and pd.notna(row.get(sasa_col)) else "NA"
        hb = row.get(hb_col, "NA") if hb_col else "NA"
        seq = row.get("sequence") or ""
        f.write(f">{nb_id}|boltzgen_rank={rank}|sasa={sasa}|hbonds={hb}\n{seq}\n")

print(f"Saved: {fasta_out}")
print(open(fasta_out).read())

## Output files

| File | Description |
|---|---|
| `nanobody_boltzgen_scores.csv` | Full ranked table with all metrics |
| `top10_boltzgen_scored.fasta` | Top 10 sequences with scores in FASTA header |
| `score_distributions.png` | Histogram of each scoring metric |
| `sasa_vs_hbonds.png` | Scatter: buried surface vs H-bonds |
| `scoring_results/` | Raw boltzgen output (CIFs, NPZs, CSVs) |

## Key metrics

| Metric | Direction | Meaning |
|---|---|---|
| `delta_sasa_refolded` | ↑ higher = better | Buried surface area at the interface |
| `plip_hbonds_refolded` | ↑ higher = better | Number of H-bonds between nanobody and target |
| `largest_hydrophobic_refolded` | ↓ lower = better | Hydrophobic patch size (developability risk) |
| `plddt` | ↑ higher = better | Per-residue folding confidence |
| `iptm` | ↑ higher = better | Interface predicted TM-score |